<a href="https://colab.research.google.com/github/juliaphaus/ds110/blob/main/JuliaHaus_DS110_S26_HW9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## (70 points total)

This homework requires the cities.db file, found where you downloaded the homework.

*Recall that AI generated answers are not permitted, but you can ask questions of an AI about Python.*

# Problem 1:  Review Problems (27 pts [10, 8, 9])

a, 10 points)  Define a class `House` that has a constructor initializing its attributes of `price` and `sqft`, and a `__str__()` method that reasonably returns a string with both pieces of information.  Then write a function `find_deals(df, thresh)` that takes a DataFrame `df` as an argument, where the columns of the DataFrame are "Price" and "Square Feet", and returns a list of House objects where Price/(Square Feet) is less than or equal to `thresh`.  The function should also modify the DataFrame to have a new Boolean column "Deal" that is True only if the house was added to the deals list to be returned; return the modified DataFrame itself as a second return value.


In [8]:
class House:
  def __init__(self, price, sqft):
    self.price = price
    self.sqft = sqft
  def __str__(self):
    return f"Price: {self.price}, Square Feet: {self.sqft}"
def find_deals(df, thresh):
  deals_list = []
  df['Deal'] = False
  for index, row in df.iterrows():
    if row['Price'] / row['Square Feet'] <= thresh:
      deals_list.append(House(row['Price'], row['Square Feet']))
      df.at[index, 'Deal'] = True
  return deals_list, df


In [9]:
import pandas as pd
df = pd.DataFrame({'Price': [800000,900000,1000000], 'Square Feet': [1000, 2000, 3000]})
deals, df = find_deals(df, 700) # Expect list with houses two and three, df with False True True final column
for deal in deals:
    print(deal)
print(df)

Price: 900000, Square Feet: 2000
Price: 1000000, Square Feet: 3000
     Price  Square Feet   Deal
0   800000         1000  False
1   900000         2000   True
2  1000000         3000   True


b, 8 pts) Write a recursive function `update(tree, d)` that takes a binary tree with attributes `left`, `right`, and `val` as attributes, and replaces all values `val` in the tree with `d[val]`.  For example, in a small tree with root value `a` and left and right children `b` and `c`, `update(root, {'a': 'z', 'b': 'y'})` would turn the tree into one with root value `z`, left child value `y`, and `c` remains the same.  Return a list of all values found in the tree that weren't in the dictionary; in this example, that would be `['c']`.  Despite this small example, your code should work with trees of arbitrary size.

In [7]:
class Tree:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

    # Quick tool for debugging to see all the values in the tree
    def __str__(self):
        s = ''
        if self.left:
            s += str(self.left) + ' '
        s += self.val
        if self.right:
            s += ' ' + str(self.right)
        return s

In [10]:
def update(tree, d):
  if tree is None:
    return []
  missing = []
  if tree.val in d:
    tree.val = d[tree.val]
  else:
    missing.append(tree.val)
  left_missing = update(tree.left, d)
  right_missing = update(tree.right, d)
  return missing + left_missing + right_missing

In [11]:
test_tree = Tree('a')
test_tree.left = Tree('b')
test_tree.right = Tree('c')
test_tree.right.right = Tree('d')
test_tree.left.left = Tree('z')
print(update(test_tree, {'a':'f', 'z': 'g'})) # Expect ['b', 'c', 'd'] (order doesn't matter)
print(test_tree) # Expect 'g b f c d' (order doesn't matter)

['b', 'c', 'd']
g b f c d


c, 9 points) Some products need to be recalled, and that means contacting the customers who bought certain serial numbers of items.  The serial numbers affected are 'ABnumberQ' where 'number' is any number of at least one digit and the 'A' and 'B' and 'Q' are literally those uppercase letters.  Some serial numbers have different, more, or fewer letters; we should ignore those.  Write a function `gather_ABQphones(filename)` which opens a json file with name `filename` and, for every key that is an affected serial number, scans the value of the key (which could contain extra junk) for a phone number of the form `XXX-XXX-XXXX` where X is a digit and either - could be missing, and adds the phone string to the list that is the return value if it's found.  If no phone number is found for a record, just move on.  Use regular expressions for both the serial number matching and the phone matching.  If an exception is thrown, you can print "bad argument" and return an empty list.

In [18]:
import json
import re
def gather_ABQphones(filename):
  phones = []
  try:
    with open(filename, 'r') as f:
      data = json.load(f)
    serial_pattern = r'^AB\d+Q$'
    phone_pattern = r'\d{3}-?\d{3}-?\d{4}'
    for serial, content in data.items():
      if re.match(serial_pattern, serial):
        match = re.search(phone_pattern, content)
        if match:
          phones.append(match.group())
  except Exception:
    return []
  return phones

In [19]:
# Creating a test file
file_dict = {
    'ABQ': '555-555-5555', # ignore
    'AB1Q': 'blah blah 123-456-7890 blah blah', # good
    'AAB1Q': '555-555-5556', # ignore
    'AB111222333Q': 'blah 123-456-7891 blah', # good
    'AB2Q': '22-333-4444', # ignore
    'AB23Q': '123-4567890' # good
}

with open('test.json', 'w') as f:
    json.dump(file_dict, f)

!cat test.json

{"ABQ": "555-555-5555", "AB1Q": "blah blah 123-456-7890 blah blah", "AAB1Q": "555-555-5556", "AB111222333Q": "blah 123-456-7891 blah", "AB2Q": "22-333-4444", "AB23Q": "123-4567890"}

In [20]:
gather_ABQphones('test.json') # Expect ['123-456-7890', '123-456-7891', '123-4567890']

['123-456-7890', '123-456-7891', '123-4567890']

# Problem 2: SQLite Queries (12 points [6,6])

For each text description of a query, make the corresponding query in SQLite (in Python).  Be sure to upload the cities.db file first.  This file contains two tables, the cities table containing "name" (of city) and "population", and the "best_cities" table, containing "city" and "reason" (one reason it's a good city, according to the AFAR website).

In [21]:
from google.colab import files

uploaded = files.upload()

Saving cities.db to cities.db


In [22]:
import sqlite3
import pandas as pd
connection = sqlite3.connect('cities.db')

(a) (6 points) Query:  city name, reason it's a good city, and population, from an inner join of the two tables.  Sort the results by city name (alphabetically, ascending) within the SQL query.  The query should produce 5 American cities.

In [23]:
query_2a = """SELECT cities.name, best_cities.reason, cities.population FROM cities INNER JOIN best_cities ON cities.name = best_cities.city ORDER BY cities.name ASC;"""
df_2a = pd.read_sql_query(query_2a, connection)
df_2a

,name,reason,population
0,Chicago,Culture,2896016
1,Las Vegas,Attractions,478434
2,Los Angeles,University,3694820
3,San Francisco,Things to do,776733
4,Washington,Attractions,572059


(b) (6 points) Find out which has the larger population in the "cities" table, the cities ending in "-ton" or the cities ending in "-ville".  Get both DataFrames, convert the population strings to integers, and find the average of each population column, then declare which is larger.




In [29]:
ton_query = "SELECT population FROM cities WHERE name LIKE '%ton'"
ville_query = "SELECT population FROM cities WHERE name LIKE '%ville'"
df_ton = pd.read_sql_query(ton_query, connection)
df_ville = pd.read_sql_query(ville_query, connection)
ton_population = pd.to_numeric(df_ton['population'])
ville_population = pd.to_numeric(df_ville['population'])
avg_ton = ton_population.mean()
avg_ville = ville_population.mean()
print(f"'-ton' cities: {avg_ton}")
print(f"'-ville' cities: {avg_ville}")


'-ton' cities: 360783.3846153846
'-ville' cities: 203525.9


**Cities ending in '-ton' have a larger population**

# Problem 3:  Advanced Pandas (15 points [3, 7, 5])

Perform the requested transformations of the table below.

In [30]:
import pandas as pd
import numpy as np

movies = pd.DataFrame({
    "Title": ["Turning Red",
              "Three Colours: Red",
              "Three Colours: Blue",
              "Three Colours: White",
              "Harold and Kumar Go to White Castle",
              "Men in Black",
              "Yellow Submarine",
              "Inception",
              "Monty Python and the Holy Grail",
              "The Triplets of Belleville",
              "Adolescence of Utena",
              "Tampopo"],
    "Genre": ["Comedy", "Drama", "Drama", "Drama", "Comedy", "Sci-Fi", "Fantasy", "Sci-Fi", "Comedy", "Comedy", "Drama", "Comedy"],
    "Year": [2022, 1994, 1994, 1994, 2004, 1997, 1968, 2010, 1975, 2003, 1999, 1985],
    "Country": ["USA", "France", "France", "France", "USA", "USA", "UK", "USA", "UK", "France", "Japan", "Japan"],
    "RTScore": [90, 100, 98, 87, 74, 92, 95, 87, 92, 94, 85, 100]
})


(a) (3 points) Use groupby to find the mean score by country.  Don't include averages of the year (use just the country and RTScore columns).

In [34]:
country_scores = movies[['Country', 'RTScore']].groupby('Country').mean()
country_scores

,RTScore
Country,
France,94.75
Japan,92.50
UK,93.50
USA,85.75


(b) (7 points) Perform an outer join with the following table on movie title.  Then set the missing values in the "Animated" column to False, the missing values in RTScore to the average of the column, and finally, display the table.

In [35]:
animated = pd.DataFrame({"Title": ["Turning Red",
                                   "Harold and Kumar Go to White Castle",
                                   "Men in Black",
                                   "Yellow Submarine",
                                   "Adolescence of Utena",
                                   "End of Evangelion",
                                   "No",
                                   "Spirited Away",
                                   "The Triplets of Belleville"],
                         "Animated": [True,False,False,True,True,True,False,True, True]})

In [40]:
merged = pd.merge(movies, animated, on="Title", how="outer")
merged["Animated"] = merged["Animated"].fillna(False)
average_score = merged["RTScore"].mean()
merged["RTScore"] = merged["RTScore"].fillna(average_score)
merged

/tmp/ipykernel_7171/2252314684.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["Animated"] = merged["Animated"].fillna(False)


,Title,Genre,Year,Country,RTScore,Animated
0,Adolescence of Utena,Drama,1999.0,Japan,85.000000,True
1,End of Evangelion,NaN,NaN,NaN,91.166667,True
2,Harold and Kumar Go to White Castle,Comedy,2004.0,USA,74.000000,False
3,Inception,Sci-Fi,2010.0,USA,87.000000,False
4,Men in Black,Sci-Fi,1997.0,USA,92.000000,False
5,Monty Python and the Holy Grail,Comedy,1975.0,UK,92.000000,False
6,No,NaN,NaN,NaN,91.166667,False
7,Spirited Away,NaN,NaN,NaN,91.166667,True
8,Tampopo,Comedy,1985.0,Japan,100.000000,False
9,The Triplets of Belleville,Comedy,2003.0,France,94.000000,True


(c) (5 points) Create a pivot table using your merged table that separately shows the average scores for every country's animated and non-animated films.  (You can leave in any default scores calculated in the previous step.)

In [45]:
pivot = merged.pivot_table(values='RTScore', index='Country', columns='Animated', aggfunc='mean')
pivot

Animated,False,True
Country,,
France,95.000000,94.0
Japan,100.000000,85.0
UK,92.000000,95.0
USA,84.333333,90.0


# Problem 4:  Graphs and Centrality (16 pts [8,8])

a, 8 pts) The following is a representation of an undirected graph with an adjacency list representation.  Write a function friends_of_friends() that can take a vertex name and an arbitrary graph like this as arguments, and return a list of all vertices that are distance 2 from the first argument.  (Do not return vertices that are distance 1 or 0, and do not repeat any vertices.)  For example, on this graph, friends_of_friends('A', graph) should return ['D', 'E'], because 'B' and 'C' are distance 1, 'D' has a distance 2 path through both, and 'E' has a distance 2 path through 'C'.

In [46]:
graph = {'A':  ['B','C'],
         'B':  ['A','C','D'],
         'C':  ['A','B','D','E'],
         'D':  ['B','C'],
         'E':  ['C','F'],
         'F':  ['E']}

In [48]:
def friends_of_friends(start_node, graph):
  dist1 = set(graph[start_node])
  friends_of_friends_set = set()
  for friend in dist1:
    neighbors = graph[friend]
    for neighbor in neighbors:
      if neighbor != start_node and neighbor not in dist1:
        friends_of_friends_set.add(neighbor)
  return list(friends_of_friends_set)

In [49]:
print(friends_of_friends('A', graph)) # expect ['D', 'E']
print(friends_of_friends('B', graph)) # expect ['E']
print(friends_of_friends('C', graph)) # expect ['F']
print(friends_of_friends('D', graph)) # expect ['A', 'E']
print(friends_of_friends('E', graph)) # expect ['A', 'B', 'D']

['E', 'D']
['E']
['F']
['E', 'A']
['B', 'A', 'D']


b, 8 pts) Write a function closeness_centrality() that calculates *approximate* closeness centrality, given a vertex name and a graph.  You can assume any distances that are greater than 2 are roughly 3 - with this assumption, you won't need a full breadth first search, but you can instead use your friends_of_friends() function to determine which nodes are 2 steps away, and the graph representation itself to tell you which nodes are one step away.  (Note that our example graph above doesn't actually have nodes at distance greater than 2 from each other.)

Some online sources have different formulas for closeness centrality, so please use our method:  find the average distance of all other vertices from the target vertex, then take the reciprocal of that average.

In [50]:
def closeness_centrality(start_node, graph):
  all_nodes = set(graph.keys())
  dist1 = set(graph[start_node])
  dist2 = set(friends_of_friends(start_node, graph))
  dist3 = all_nodes - dist1 - dist2 - {start_node}
  total_dist = (len(dist1) * 1) + (len(dist2) * 2) + (len(dist3) * 3)
  number_other_nodes = len(all_nodes) - 1
  average_dist = total_dist / number_other_nodes
  return 1 / average_dist

In [51]:
print(closeness_centrality('C', graph)) # expect 0.833
print(closeness_centrality('D', graph)) # expect 0.556

0.8333333333333334
0.5555555555555556


**When you are done, submit both your .ipynb (File->Download->Download .ipynb in Google Colab) and a PDF (File->Print->Save to PDF) to Blackboard where you found this assignment.**  (The backup PDF helps us give you points if there's a problem with your .ipynb.)